In [ ]:
import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import silhouette_score
import os
sc.settings.set_figure_params(dpi=150, facecolor="white")
sc.settings.figdir = r"C:\Users\taylorhearn\Desktop\scRNAseq_Dorota\Green_Removed"
conditions = ["AAG05732_WS_Blue", "AAG05733_WW_Red", "AAG05734_WP_Yellow"]  #, "AAG05735_PP_Green"]

# Load Data
- Add experimental info and get in the same format as Agus

In [ ]:
# Read each 10x filtered matrix, keyed by its sample name.
DATA_DIR = r"C:\Users\taylorhearn\Desktop\scRNAseq_Dorota\Filtered_Matrices-2"
samples = {
    name: sc.read_10x_mtx(os.path.join(DATA_DIR, name), var_names="gene_symbols", cache=True)
    for name in conditions
}

In [ ]:
# Concatenate the 4 samples. label="sample" records each cell's sample of origin
# (dict keys become the labels); index_unique makes barcodes unique across samples.
adata = sc.concat(samples, axis=0, label="sample", index_unique="_", join="outer")
adata.obs_names_make_unique()

# No technical replicates: each of the 4 samples IS its own biological condition.
adata.obs["condition"] = adata.obs["sample"].astype("category")

In [ ]:
adata.obs_keys()

# QC metrics and filtering

The Parse `all-sample` matrix is **unfiltered** (~2 million background barcodes).
Compute QC metrics, inspect the plots, then filter down to real cells.
Gene prefixes are **human** (`MT-`, `RPS`/`RPL`).

In [ ]:
# # Human gene prefixes
adata.var["mt"]   = adata.var_names.str.startswith("MT-") #new col that labels mitochondrial genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL")) # new col that labels ribosomal genes
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo"], inplace=True, log1p=True) # adds a load more columns

In [ ]:
sc.pl.scatter(adata, x="total_counts", y="pct_counts_mt", save="initial_qc_scatter_mt.png")
sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts", save="initial_qc_scatter_genes.png")
sc.pl.violin(adata, ["log1p_n_genes_by_counts", "log1p_total_counts", "pct_counts_mt", "pct_counts_ribo"], jitter=0.4, multi_panel=True, save="initial_qc_violin.png")

In [ ]:
# Save pre-filter checkpoint (adata with QC metrics, unfiltered).
# Run this once after the violin plots look reasonable.
# Next time, skip the read_parse + calculate_qc_metrics cells and just reload here:
# prefilt_path = r"z:\Bel\Marina_Side_Projects\scRNA\adata_prefilt.h5ad"
# adata = sc.read_h5ad(prefilt_path)

# adata.write_h5ad(prefilt_path)
# print(f"Saved pre-filter checkpoint: {adata.shape} → {prefilt_path}")


In [ ]:
sc.pp.filter_cells(adata, min_genes=100)      # drop near-empty barcodes, Agus had 100
sc.pp.filter_cells(adata, min_counts = 500)
sc.pp.filter_genes(adata, min_cells=3)        # drop genes seen in <3 cells, same as Agus

keep = (
    (adata.obs["n_genes_by_counts"] < 6500)   # upper bound: likely doublets, same as Agus
    & (adata.obs["total_counts"] < 50000)     # upper bound on library size
    & (adata.obs["pct_counts_mt"] < 15)       # high mito % = dying cells, Agus = 20
    & (adata.obs["pct_counts_ribo"] < 15)     # very high ribo % = low-info cells, same as Agus
)
filtered_data = adata[keep].copy()
print(filtered_data)
print(filtered_data.shape)

sc.pl.violin(filtered_data, ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo"], jitter=0.4, multi_panel=True, save="post_filter_qc_violin.png")

#### Check How Much of Each Sample Has Been Filtered

In [ ]:
filtered_counts = filtered_data.obs["sample"].value_counts()
original_counts = adata.obs['sample'].value_counts()
total_counts = pd.DataFrame({"original": original_counts, "filtered": filtered_counts})
total_counts["filtered_fraction"] = total_counts["filtered"] / total_counts["original"]
# total_counts.to_csv(r"C:\Users\taylorhearn\Desktop\scRNA\total_counts_15pct_mitocutoff.csv")

In [ ]:
total_counts

In [ ]:
# FL43_Chip = adata.obs[adata.obs["sample"] == "FL43_Chip"]
# FL43_Chip
# sns.histplot(data=adata.obs[adata.obs["condition"]=="FL_Chip_plus"], x="pct_counts_mt", hue="sample", bins=50, element="step", stat="density", common_norm=False)

In [ ]:
# sns.jointplot(data=filtered_data.obs, x="total_counts", y="n_genes_by_counts", kind="hex")
# sns.jointplot(data=filtered_data.obs, x="log1p_total_counts", y="log1p_n_genes_by_counts", kind="hex")

# sc.pl.scatter(filtered_data, x="total_counts", y="pct_counts_mt", color="pct_counts_ribo", color_map="viridis", size=50, alpha=0.5)
# sc.pl.scatter(filtered_data, x="total_counts", y="n_genes_by_counts", color="pct_counts_ribo")

In [ ]:
# Save filtered adata so you can reload it directly next time.
# To skip all the reading + QC + filtering: adata = sc.read_h5ad(filtered_path)
# filtered_path = r"z:\Bel\Marina_Side_Projects\scRNA\adata_filtered.h5ad"
# adata.write_h5ad(filtered_path)
# print(f"Saved filtered data: {adata.shape} → {filtered_path}")
# adata = sc.read_h5ad(filtered_path)


# Normalisation, Scaling and PCA
- Normalise once on the full combined object. normalize_total puts every cell on the same total-count budget (e.g. counts per 10k), so you're comparing proportions of expression, not raw magnitudes.
- Then we reduce ~2000 genes -> ~50 principal components (which still contain technical/sample differences)

In [ ]:
# Keep raw counts, then normalize once on the full combined object
filtered_data.layers["counts"] = filtered_data.X.copy()
sc.pp.normalize_total(filtered_data)
sc.pp.log1p(filtered_data)
filtered_data.raw = filtered_data

In [ ]:
# finds the 2000 genes that vary most across the cells, computed per sample and then combined so no single sample dominates
# it just flags the genes in filtered_data.var["highly_variable"] (i.e. "highly variable = True")
sc.pp.highly_variable_genes(filtered_data, n_top_genes=2000, batch_key="sample")
# puts genes on a comparable scale and clips extreme values so a few highly-expressed genes don't dominate
sc.pp.scale(filtered_data, max_value=10, zero_center=False)
#compress the highly variable genes into a smaller subset of prinicipal components (default = 50). This X_pca is the numerical summary every later step builds on
sc.tl.pca(filtered_data, use_highly_variable=True)
# this stage isn't plottable, it only produces coordinates and statistics.

In [ ]:
sc.pl.highly_variable_genes(filtered_data) # shows which genes were selected (black) vs not (grey) by mean expression vs dispersion
sc.pl.pca_variance_ratio(filtered_data, n_pcs=50, log=True) # how much variance each PC captures. Use it to decide how many PCs are worth keeping (where the curve flattens)
sc.pl.pca(filtered_data, color="sample", dimensions=[(0,1)], ncols=1, size=3) # a quick look at the PCA plot, colored by sample. If you see clear separation by sample, you may need to run Harmony to integrate them before clustering.
sc.pl.pca(filtered_data, color="sample", dimensions=[(2,3)], ncols=1, size=3) # a quick look at the PCA plot, colored by sample. If you see clear separation by sample, you may need to run Harmony to integrate them before clustering.

### Integration, UMAP and clustering

Harmony removes technical differences between samples so the same cell types overlap
across conditions. This - not normalization - is what makes the 7 conditions comparable.
- Take the PCs we calculated the in the cell above and correct batch effects (harmony_integrate -> X_pca_harmony)
- Harmony removes technical/sample differences so the same cell type from difference samples overlaps
- "make conditions comparable, then visualise and group cells"

In [ ]:
# Batch-correct across samples (requires harmonypy: pip install harmonypy)
# GPU acceleration via rapids_singlecell is Linux-only (RAPIDS has no Windows wheels).
# To use GPU: run this notebook inside WSL2 with CUDA support.
sc.external.pp.harmony_integrate(filtered_data, key="sample")
sc.pp.neighbors(filtered_data, use_rep="X_pca_harmony", n_neighbors=30)
sc.tl.umap(filtered_data, random_state=42)


# Choose The Number of Clusters

In [ ]:
# Scan a range of Leiden resolutions on the existing Harmony neighbour graph.
# Each run stores its labels in adata.obs[f"leiden_{res}"] so we can compare without recomputing.
resolutions = [0.2, 0.4, 0.6, 0.8, 1.0]
for res in resolutions:
    sc.settings.figdir = r"C:\Users\taylorhearn\Desktop\scRNAseq_Dorota\Figures\{}".format(str(res).replace(".", "_"))
    sc.tl.leiden(filtered_data, flavor="leidenalg", n_iterations=2, resolution=res, key_added=f"leiden_{res}", random_state=42, directed=False)

n_clusters = {res: filtered_data.obs[f"leiden_{res}"].nunique() for res in resolutions}
for res, n in n_clusters.items():
    print(f"resolution {res}: {n} clusters")


# UMAP grid: one panel per resolution. Watch for the point where clusters start
# fragmenting into many tiny, hard-to-interpret islands.
sc.pl.umap(
    filtered_data,
    color=[f"leiden_{res}" for res in resolutions],
    legend_loc="on data",
    legend_fontsize=6,
    ncols=3,
    wspace=0.25,
    title=[f"res {res} ({n_clusters[res]} clusters)" for res in resolutions],
)

# Quantitative guides for picking a resolution.
# (1) Cluster count vs resolution - a plateau means structure is stable.
# (2) Silhouette score on a subsample of the Harmony PCs - higher = better separated.

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(list(n_clusters.keys()), list(n_clusters.values()), "o-")
ax.set_xlabel("Leiden resolution")
ax.set_ylabel("# clusters")
ax.set_title("Cluster count vs resolution")
ax.grid(True, alpha=0.3)
plt.show()

rng = np.random.default_rng(0)
idx = rng.choice(filtered_data.n_obs, size=min(10000, filtered_data.n_obs), replace=False)
X_sub = filtered_data.obsm["X_pca_harmony"][idx]
print("Silhouette score (subsampled 10k cells, higher = better separated):")
for res in resolutions:
    labels = filtered_data.obs[f"leiden_{res}"].values[idx]
    if len(np.unique(labels)) > 1:
        score = silhouette_score(X_sub, labels)
        print(f"  res {res}: {score:.3f}  ({n_clusters[res]} clusters)")


# Full Analysis Per Resolution

Run the complete downstream analysis (UMAPs, marker genes, canonical-marker annotation,
composition across conditions) for **every** Leiden resolution. All outputs for a given
resolution are written into a dedicated subfolder named after that resolution, e.g.
`Figures/res_0_6/`, so each resolution has its own self-contained set of figures, CSVs
and a saved `.h5ad`.

In [ ]:


# Resolutions to fully analyse (must match those scanned above, or any subset).
resolutions = [0.2, 0.4, 0.6, 0.8]

# Base locations. Each resolution gets its own subfolder underneath BASE_FIGDIR.
BASE_FIGDIR = r"C:\Users\taylorhearn\Desktop\scRNAseq_Dorota\Figures"
OUTPUT_DIR  = r"C:\Users\taylorhearn\Desktop\scRNAseq_Dorota\Output"


immune_markers = {
    # "B cell": ["CD19", "MS4A1", "CD79A", "CD79B"],
    # "Germinal center B": ["BCL6", "AICDA", "RGS13", "LMO2"],
    # "FL B cell": ["BCL2", "MME", "CD10"],          # t(14;18) BCL2+ CD10+ GC-derived (follicular lymphoma)
    # "Plasma": ["MZB1", "XBP1", "PRDM1", "SDC1"],
    # "T cell": ["CD3D", "CD3E", "CD2"],
    # "CD4 T": ["CD4", "IL7R"],
    # "CD8 T": ["CD8A", "CD8B", "GZMK"],
    # "Tfh": ["CXCL13", "PDCD1", "ICOS"],
    # "NK": ["NKG7", "GNLY", "KLRD1"],
    # "Myeloid/DC": ["LYZ", "CD68", "ITGAX"],
    "Endothelial": ["PECAM1", "VWF", "CDH5", "CLDN5"],          # blood vessels
    # "Lymphatic endothelial": ["PROX1", "LYVE1", "PDPN"],       # lymphatic sinuses
    "Fibroblast/stromal": ["PDGFRA", "PDGFRB", "COL1A1", "COL3A1", "DCN", "LUM"],
    # "FRC": ["ACTA2", "PDPN", "CCL19", "CCL21"],                 # fibroblastic reticular cells
    # "FDC": ["CR2", "CR1", "FDCSP", "CLU"],                      # follicular dendritic cells (germinal centers)
    "Proliferating": ["MKI67", "TOP2A"],
}


def highlight_umap(adata, groupby, size=10, ncols=4, na_color="lightgrey", savefig=None):
    """Highlight one category at a time on the full UMAP (others greyed out)."""
    groups = list(adata.obs[groupby].cat.categories) if hasattr(adata.obs[groupby], "cat") \
        else sorted(adata.obs[groupby].unique())
    nrows = int(np.ceil(len(groups) / ncols))
    fig, axes = plt.subplots(
        nrows=nrows, ncols=ncols,
        figsize=(4 * ncols, 4 * nrows),
        gridspec_kw={"wspace": 0.3, "hspace": 0.3},
    )
    axes = np.atleast_1d(axes).ravel()
    for ax, group in zip(axes, groups):
        sc.pl.umap(
            adata, color=groupby, groups=[group], na_color=na_color,
            size=size, title=str(group), legend_loc=None, frameon=False, show=False, ax=ax,
        )
    for ax in axes[len(groups):]:
        ax.axis("off")
    if savefig:
        plt.savefig(savefig, bbox_inches="tight")
    plt.show()

In [ ]:
def analyze_resolution(adata, res, base_figdir=BASE_FIGDIR, output_dir=OUTPUT_DIR):
    """Run the full downstream analysis for a single Leiden resolution and save every
    output into a per-resolution subfolder (Figures/res_<res>/)."""
    res_tag = str(res).replace(".", "_")
    figdir = os.path.join(base_figdir, f"res_{res_tag}")
    os.makedirs(figdir, exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)   # <-- add this so the .h5ad write target exists
    sc.settings.figdir = figdir
    # Reuse the labels from the resolution scan if present, otherwise compute them.
    key = f"leiden_{res}"
    if key not in adata.obs:
        sc.tl.leiden(adata, flavor="leidenalg", n_iterations=2, resolution=res,
                     key_added=key, random_state=42, directed=False)
    adata.obs["clusters"] = adata.obs[key].astype("category")
    n = adata.obs["clusters"].nunique()
    print(f"\n=== resolution {res}: {n} clusters -> {figdir} ===")

    # Recompute the dendrogram for THIS resolution's clusters up front, so every plot
    # below that defaults to dendrogram=True uses one matching the current cluster count.
    sc.tl.dendrogram(adata, groupby="clusters")

    # 1. Clusters + whether Harmony mixed samples/conditions, plus QC sanity overlay.
    sc.pl.umap(adata, color=["clusters", "sample", "condition"], wspace=0.4, ncols=3,
               save="_overview.png", show=True)
    sc.pl.umap(adata, color=["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo"],
               wspace=0.4, ncols=3, save="_qc.png", show=True)

    # 2. Publication-style UMAPs.
    sc.pl.umap(adata, color="clusters", legend_loc="on data", frameon=False,
               title=f"Leiden clusters (res {res})", save="_leiden.png", show=True)
    sc.pl.umap(adata, color="condition", frameon=False, title="Condition",
               save="_condition.png", show=True)

    # 3. Highlight one sample / condition at a time.
    highlight_umap(adata, "sample", size=10, ncols=4,
                   savefig=os.path.join(figdir, "umap_sample_highlight.png"))
    highlight_umap(adata, "condition", size=10, ncols=4,
                   savefig=os.path.join(figdir, "umap_condition_highlight.png"))

    # 4. Data-driven marker genes per cluster.
    sc.tl.rank_genes_groups(adata, groupby="clusters", method="wilcoxon", use_raw=True, rankby_abs=True)
    sc.pl.rank_genes_groups_dotplot(
        adata, n_genes=5, values_to_plot="logfoldchanges", cmap="bwr",
        vmin=-4, vmax=4, colorbar_title="log fold change",
        save="_markers.png", show=True,
    )
    markers = sc.get.rank_genes_groups_df(adata, group=None)
    markers.to_csv(os.path.join(figdir, "cluster_markers.csv"), index=False)

    # 5. Canonical marker annotation (drop genes absent from this dataset).
    present = {k: [g for g in v if g in adata.var_names] for k, v in immune_markers.items()}
    present = {k: v for k, v in present.items() if v}
    # Recompute the dendrogram for THIS resolution's clusters. It is cached in
    # adata.uns and would otherwise be stale (wrong # categories) from a prior resolution.
    sc.tl.dendrogram(adata, groupby="clusters")
    sc.pl.dotplot(adata, present, groupby="clusters", standard_scale="var",
                  dendrogram=True, save="_canonical_markers.png", show=True)
    key_genes = [g for g in ["MS4A1", "CD3D", "CD8A", "MZB1", "LYZ", "MKI67"] if g in adata.var_names]
    sc.pl.umap(adata, color=key_genes, ncols=3, cmap="viridis", use_raw=True,
               save="_key_genes.png", show=True)

    # 6. Cluster composition across conditions.
    comp = pd.crosstab(adata.obs["condition"], adata.obs["clusters"], normalize="index")
    ax = comp.plot(kind="bar", stacked=True, figsize=(8, 4), colormap="tab20", width=0.85)
    ax.set_ylabel("Fraction of cells")
    ax.set_xlabel("Condition")
    ax.set_title(f"Cluster composition per condition (res {res})")
    ax.legend(title="cluster", bbox_to_anchor=(1.01, 1), loc="upper left", ncol=2, fontsize=7)
    plt.tight_layout()
    plt.savefig(os.path.join(figdir, "composition_per_condition.png"), bbox_inches="tight")
    plt.show()
    comp.round(3).to_csv(os.path.join(figdir, "composition_per_condition.csv"))

    # 7. Persist the clustered object for this resolution.
    out_path = os.path.join(output_dir, f"adata_clustered_res_{res_tag}.h5ad")
    adata.write_h5ad(out_path)
    print(f"Saved clustered object: {adata.shape} -> {out_path}")

In [ ]:
# Run the full analysis for every resolution. Each one saves into its own subfolder.
for res in resolutions:
    analyze_resolution(filtered_data, res)